# Preprocessing

## Set up

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from typing import Tuple, Dict, List, Optional
from dataclasses import dataclass
from tqdm import tqdm
import geopandas as gpd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from functools import partial
import multiprocessing as mp
from numba import jit, prange
import joblib
import warnings
import os
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================
@dataclass
class Config:
    """Centralized configuration for data processing"""
    # File paths
    mekong_csv_path: str = "../data/raw/mekong_data_final_2019_2024.csv"
    sentinel_2_csv_path: str = "../data/raw/sentinel2_extract_data.csv"
    sentinel_1_csv_path: str = "../data/raw/sentinel1_extract_data.csv"
    target_csv_path: str = "../data/raw/target_precipitation.csv"
    vn_adm1_path: str = '../data/raw/gadm41_VNM_1.json'

    # Date ranges
    train_start: str = "2019-01-01"
    train_end: str = "2023-12-31"
    val_start: str = "2024-01-01"
    val_end: str = "2024-06-30"
    test_start: str = "2024-07-01"
    test_end: str = "2024-12-31"

    # Spatial parameters
    min_lon: float = 103.5
    max_lon: float = 107.0
    min_lat: float = 8.0
    max_lat: float = 11.5
    resolution: float = 0.1

    # Performance parameters
    n_workers: int = None
    chunk_size: int = 50  # Reduced for better memory usage
    use_multiprocessing: bool = True

    # Columns to exclude from scaling
    exclude_cols: List[str] = None

    # PCA feature groups
    pca_groups: List[List[str]] = None

    # Mekong provinces
    mekong_provinces: List[str] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = max(1, mp.cpu_count() - 1)

        if self.exclude_cols is None:
            self.exclude_cols = ['minx', 'maxx', 'miny', 'maxy', 'province', 'date', 'geometry', 'id']

        if self.pca_groups is None:
            self.pca_groups = [
                ['Rainf_tavg', 'Rainf_f_tavg', 'CanopInt_inst', 'ECanop_tavg', 'Lwnet_tavg'],
                ['SoilMoi0_10cm_inst', 'Qair_f_inst', 'SoilMoi10_40cm_inst', 'volumetric_soil_water_layer_3', 'dewpoint_temperature_2m']
            ]

        if self.mekong_provinces is None:
            self.mekong_provinces = [
                "LongAn", "TiềnGiang", "BếnTre", "TràVinh", "VĩnhLong",
                "ĐồngTháp", "AnGiang", "CầnThơ", "HậuGiang",
                "SócTrăng", "BạcLiêu", "CàMau", "KiênGiang"
            ]


# ============================================================================
# NUMBA OPTIMIZED FUNCTIONS
# ============================================================================
@jit(nopython=True, parallel=True)
def fill_mekong_tensor_numba(feature_tensor, count_tensor, t_indices,
                             x_starts, x_ends, y_starts, y_ends, feat_vals):
    """Numba-optimized tensor filling for Mekong data"""
    n_rows = len(t_indices)

    for i in prange(n_rows):
        t = t_indices[i]
        x0, x1 = x_starts[i], x_ends[i]
        y0, y1 = y_starts[i], y_ends[i]

        for y in range(y0, y1):
            for x in range(x0, x1):
                for c in range(feat_vals.shape[1]):
                    feature_tensor[t, y, x, c] += feat_vals[i, c]
                    count_tensor[t, y, x, c] += 1

    return feature_tensor, count_tensor


@jit(nopython=True)
def compute_sentinel_slice(dates_array, target_date, x_starts, x_ends,
                          y_starts, y_ends, feat_vals, H, W, C):
    """Numba-optimized computation for single time slice"""
    best_td = np.full((H, W, C), np.inf, dtype=np.float32)
    best_sum = np.zeros((H, W, C), dtype=np.float32)
    best_count = np.zeros((H, W, C), dtype=np.int32)

    n_rows = len(dates_array)

    for i in range(n_rows):
        td = abs(dates_array[i] - target_date)
        x0, x1 = x_starts[i], x_ends[i]
        y0, y1 = y_starts[i], y_ends[i]

        for y in range(y0, y1):
            for x in range(x0, x1):
                for c in range(C):
                    v = feat_vals[i, c]
                    if not np.isnan(v):
                        if td < best_td[y, x, c]:
                            best_td[y, x, c] = td
                            best_sum[y, x, c] = v
                            best_count[y, x, c] = 1
                        elif abs(td - best_td[y, x, c]) < 0.001:
                            best_sum[y, x, c] += v
                            best_count[y, x, c] += 1

    result = np.zeros((H, W, C), dtype=np.float32)
    for y in range(H):
        for x in range(W):
            for c in range(C):
                if best_count[y, x, c] > 0:
                    result[y, x, c] = best_sum[y, x, c] / best_count[y, x, c]

    return result


def process_sentinel_time_chunk(chunk_indices, time_array, dates_array,
                                x_starts, x_ends, y_starts, y_ends,
                                feat_vals, H, W, C):
    """Process a chunk of time indices - MUST BE TOP-LEVEL FOR MULTIPROCESSING"""
    chunk_result = np.zeros((len(chunk_indices), H, W, C), dtype=np.float32)
    for i, t_idx in enumerate(chunk_indices):
        target_date = time_array[t_idx]
        chunk_result[i] = compute_sentinel_slice(
            dates_array, target_date, x_starts, x_ends,
            y_starts, y_ends, feat_vals, H, W, C
        )
    return chunk_result, chunk_indices


def compute_sentinel_slice(dates_array, target_date,
                           x_starts, x_ends, y_starts, y_ends,
                           feat_vals, H, W, C):

    best_td   = np.full((H, W, C), np.inf, dtype=np.float32)
    best_sum  = np.zeros((H, W, C), dtype=np.float32)
    best_count = np.zeros((H, W, C), dtype=np.int32)

    n_rows = len(dates_array)

    for i in range(n_rows):
        td = abs(dates_array[i] - target_date)
        x0, x1 = x_starts[i], x_ends[i]
        y0, y1 = y_starts[i], y_ends[i]

        for y in range(y0, y1):
            for x in range(x0, x1):
                for c in range(C):
                    v = feat_vals[i, c]
                    if not np.isnan(v):
                        if td < best_td[y, x, c]:
                            best_td[y, x, c] = td
                            best_sum[y, x, c] = v
                            best_count[y, x, c] = 1
                        elif abs(td - best_td[y, x, c]) < 0.001:
                            best_sum[y, x, c] += v
                            best_count[y, x, c] += 1

    result = np.zeros((H, W, C), dtype=np.float32)
    for y in range(H):
        for x in range(W):
            for c in range(C):
                if best_count[y, x, c] > 0:
                    result[y, x, c] = best_sum[y, x, c] / best_count[y, x, c]

    return result



# ============================================================================
# DATA LOADER (OPTIMIZED)
# ============================================================================
class DataLoader:
    """Handle loading and initial preprocessing of CSV files"""

    def __init__(self, config: Config):
        self.config = config

    def load_all_data(self) -> Dict[str, pd.DataFrame]:
        """Load all CSV files in parallel"""
        print("Loading data files in parallel...")

        with ThreadPoolExecutor(max_workers=4) as executor:
            future_mekong = executor.submit(self._load_mekong)
            future_sentinel_2 = executor.submit(self._load_sentinel_2)
            future_sentinel_1 = executor.submit(self._load_sentinel_1)
            future_target = executor.submit(self._load_target)

            df_mekong = future_mekong.result()
            df_sentinel_2 = future_sentinel_2.result()
            df_sentinel_1 = future_sentinel_1.result()
            df_target = future_target.result()

        return {
            'mekong': df_mekong,
            'sentinel_1': df_sentinel_1,
            'sentinel_2': df_sentinel_2,
            'target': df_target
        }

    def _load_mekong(self) -> pd.DataFrame:
        df = pd.read_csv(self.config.mekong_csv_path)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        return df

    def _load_target(self) -> pd.DataFrame:
        df = pd.read_csv(self.config.target_csv_path)
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        return df

    def _load_sentinel_2(self) -> pd.DataFrame:
        df = pd.read_csv(self.config.sentinel_2_csv_path)
        df['date'] = pd.to_datetime(df['image_date'], format='%Y-%m-%d_%H-%M-%S', errors='coerce')
        df.drop('image_date', axis=1, inplace=True)
        return df

    def _load_sentinel_1(self) -> pd.DataFrame:
        df = pd.read_csv(self.config.sentinel_1_csv_path)
        df = df.drop(columns=['Total_Backscatter_mean', '_source_file'], errors='ignore')
        df['date'] = pd.to_datetime(df['datetime'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
        df.drop('datetime', axis=1, inplace=True, errors='ignore')
        return df


# ============================================================================
# DATA SPLITTER
# ============================================================================
class DataSplitter:
    """Split data into train/val/test sets"""

    def __init__(self, config: Config):
        self.config = config
        self.train_start = pd.Timestamp(config.train_start)
        self.train_end = pd.Timestamp(config.train_end)
        self.val_start = pd.Timestamp(config.val_start)
        self.val_end = pd.Timestamp(config.val_end)
        self.test_start = pd.Timestamp(config.test_start)
        self.test_end = pd.Timestamp(config.test_end)

    def split(self, df: pd.DataFrame, date_col: str = 'date') -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Split dataframe by date ranges using optimized boolean indexing"""
        date_series = df[date_col]

        train_mask = (date_series >= self.train_start) & (date_series <= self.train_end)
        val_mask = (date_series >= self.val_start) & (date_series <= self.val_end)
        test_mask = (date_series >= self.test_start) & (date_series <= self.test_end)

        return df[train_mask].copy(), df[val_mask].copy(), df[test_mask].copy()

    def split_all(self, data_dict: Dict[str, pd.DataFrame]) -> Dict[str, Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]]:
        """Split all datasets"""
        print("Splitting data into train/val/test...")
        result = {}
        for name, df in data_dict.items():
            train, val, test = self.split(df)
            result[name] = (train, val, test)
            print(f"{name:12s} - Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")
        return result


# ============================================================================
# DATA PROCESSOR
# ============================================================================
class MekongDataProcessor:
    """Handle scaling and PCA transformations"""

    def __init__(self, config: Config):
        self.config = config
        self.robust_scaler = RobustScaler()
        self.robust_scaler_target = RobustScaler()

    def _get_numeric_columns(self, df: pd.DataFrame) -> List[str]:
        """Get numeric columns excluding specific ones"""
        num_cols = df.select_dtypes(include=['number']).columns.tolist()
        return [c for c in num_cols if c not in self.config.exclude_cols]

    def robust_scaling(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame,
        df_test: pd.DataFrame,
        target: bool = False
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Apply robust scaling to numeric columns"""
        num_cols = self._get_numeric_columns(df_train)

        if not num_cols:
            return df_train.copy(), df_val.copy(), df_test.copy()

        scaler = self.robust_scaler_target if target else self.robust_scaler

        df_train_scaled = df_train.copy()
        df_val_scaled = df_val.copy()
        df_test_scaled = df_test.copy()

        df_train_scaled[num_cols] = scaler.fit_transform(df_train[num_cols])
        df_val_scaled[num_cols] = scaler.transform(df_val[num_cols])
        df_test_scaled[num_cols] = scaler.transform(df_test[num_cols])

        return df_train_scaled, df_val_scaled, df_test_scaled

    def apply_pca(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame,
        df_test: pd.DataFrame,
        feature_list: List[str],
        new_col_prefix: str,
        n_components: int = 1
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Apply PCA to a feature group"""
        available_features = [f for f in feature_list if f in df_train.columns]
        if not available_features:
            return df_train, df_val, df_test

        pca = PCA(n_components=n_components, random_state=42)

        train_feats = df_train[available_features].select_dtypes(include=[np.number])
        val_feats = df_val[available_features].select_dtypes(include=[np.number])
        test_feats = df_test[available_features].select_dtypes(include=[np.number])

        if train_feats.shape[1] == 0:
            return df_train, df_val, df_test

        pca_train = pca.fit_transform(train_feats)
        pca_val = pca.transform(val_feats)
        pca_test = pca.transform(test_feats)

        df_train_new = df_train.copy()
        df_val_new = df_val.copy()
        df_test_new = df_test.copy()

        for i in range(n_components):
            col_name = f"{new_col_prefix}_PC{i+1}"
            df_train_new[col_name] = pca_train[:, i]
            df_val_new[col_name] = pca_val[:, i]
            df_test_new[col_name] = pca_test[:, i]

        df_train_new.drop(columns=available_features, inplace=True, errors='ignore')
        df_val_new.drop(columns=available_features, inplace=True, errors='ignore')
        df_test_new.drop(columns=available_features, inplace=True, errors='ignore')

        return df_train_new, df_val_new, df_test_new

    def apply_pca_groups(
        self,
        df_train: pd.DataFrame,
        df_val: pd.DataFrame,
        df_test: pd.DataFrame,
        n_components: int = 2
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Apply PCA to multiple feature groups"""
        print("Applying PCA to feature groups...")
        for feature_list in self.config.pca_groups:
            prefix = "_".join(feature_list[:2])
            df_train, df_val, df_test = self.apply_pca(
                df_train, df_val, df_test, feature_list, prefix, n_components
            )
        return df_train, df_val, df_test


# ============================================================================
# TENSOR BUILDER (HEAVILY OPTIMIZED WITH NUMBA)
# ============================================================================
class TensorBuilder:
    """Build spatial-temporal tensors from dataframes with Numba optimization"""

    def __init__(self, config: Config):
        self.config = config
        self.lon_bins = np.arange(config.min_lon, config.max_lon, config.resolution)
        self.lat_bins = np.arange(config.min_lat, config.max_lat, config.resolution)
        self.H = len(self.lat_bins)
        self.W = len(self.lon_bins)

    def build_mekong_tensor(self, df: pd.DataFrame, features: List[str]) -> np.ndarray:
        """Build tensor for Mekong/target data with bbox - NUMBA ACCELERATED"""
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])

        unique_dates = sorted(df['date'].unique())
        date_to_idx = {date: idx for idx, date in enumerate(unique_dates)}

        T, C = len(unique_dates), len(features)

        # Pre-allocate tensors
        feature_tensor = np.zeros((T, self.H, self.W, C), dtype=np.float32)
        count_tensor = np.zeros((T, self.H, self.W, C), dtype=np.int32)

        # Vectorized computation of indices
        df['t_idx'] = df['date'].map(date_to_idx)
        df['x_start'] = np.clip(np.digitize(df['minx'].values, self.lon_bins) - 1, 0, self.W - 1)
        df['x_end'] = np.clip(np.digitize(df['maxx'].values, self.lon_bins), 0, self.W)
        df['y_start'] = np.clip(np.digitize(df['miny'].values, self.lat_bins) - 1, 0, self.H - 1)
        df['y_end'] = np.clip(np.digitize(df['maxy'].values, self.lat_bins), 0, self.H)

        # Filter valid bboxes
        valid_mask = (df['x_end'] > df['x_start']) & (df['y_end'] > df['y_start'])
        df = df[valid_mask]

        # Convert to numpy arrays for Numba
        t_indices = df['t_idx'].values.astype(np.int32)
        x_starts = df['x_start'].values.astype(np.int32)
        x_ends = df['x_end'].values.astype(np.int32)
        y_starts = df['y_start'].values.astype(np.int32)
        y_ends = df['y_end'].values.astype(np.int32)
        feat_vals = df[features].values.astype(np.float32)

        # Use Numba-optimized function
        feature_tensor, count_tensor = fill_mekong_tensor_numba(
            feature_tensor, count_tensor, t_indices,
            x_starts, x_ends, y_starts, y_ends, feat_vals
        )

        # Vectorized average
        with np.errstate(divide='ignore', invalid='ignore'):
            tensor = np.where(count_tensor > 0, feature_tensor / count_tensor, 0)

        return tensor

    def build_sentinel_tensor(
        self,
        df: pd.DataFrame,
        features: List[str],
        begin_year: int,
        end_year: int,
        subset: str = 'train'
    ) -> np.ndarray:
        """Build tensor for Sentinel data - OPTIMIZED WITH PARALLEL PROCESSING"""
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])

        C = len(features)
        time_index = self._get_time_index(begin_year, end_year, subset)
        T = len(time_index)

        # Precompute spatial indices
        df = self._precompute_spatial_indices(df, features)

        # Convert to numpy arrays
        dates_array = (df['date'].values.astype('datetime64[D]') -
                      np.datetime64('1970-01-01')).astype(np.float32)
        time_array = (time_index.values.astype('datetime64[D]') -
                     np.datetime64('1970-01-01')).astype(np.float32)

        x_starts = df['x_start'].values.astype(np.int32)
        x_ends = df['x_end'].values.astype(np.int32)
        y_starts = df['y_start'].values.astype(np.int32)
        y_ends = df['y_end'].values.astype(np.int32)
        feat_vals = df[features].values.astype(np.float32)

        # Result tensor
        tensor = np.zeros((T, self.H, self.W, C), dtype=np.float32)

        # Use parallel processing with multiprocessing
        if self.config.use_multiprocessing and T > 100:
            print(f"Building Sentinel tensor ({subset}) with multiprocessing...")

            # Split work into chunks
            chunks = [list(range(i, min(i + self.config.chunk_size, T)))
                     for i in range(0, T, self.config.chunk_size)]

            # Create partial function with fixed arguments
            process_func = partial(
                process_sentinel_time_chunk,
                time_array=time_array,
                dates_array=dates_array,
                x_starts=x_starts,
                x_ends=x_ends,
                y_starts=y_starts,
                y_ends=y_ends,
                feat_vals=feat_vals,
                H=self.H,
                W=self.W,
                C=C
            )

            with ProcessPoolExecutor(max_workers=self.config.n_workers) as executor:
                futures = [executor.submit(process_func, chunk) for chunk in chunks]

                for future in tqdm(futures, desc=f"Processing {subset}"):
                    chunk_result, chunk_indices = future.result()
                    for i, t_idx in enumerate(chunk_indices):
                        tensor[t_idx] = chunk_result[i]
        else:
            # Sequential processing with progress bar
            for t_idx in tqdm(range(T), desc=f"Building Sentinel tensor ({subset})"):
                target_date = time_array[t_idx]
                tensor[t_idx] = compute_sentinel_slice(
                    dates_array, target_date, x_starts, x_ends,
                    y_starts, y_ends, feat_vals, self.H, self.W, C
                )

        return tensor

    def _precompute_spatial_indices(self, df: pd.DataFrame, features: List[str]) -> pd.DataFrame:
        """Precompute all spatial indices"""
        df = df.copy()

        df['x_start'] = np.clip(np.digitize(df['minx'].values, self.lon_bins) - 1, 0, self.W - 1)
        df['x_end'] = np.clip(np.digitize(df['maxx'].values, self.lon_bins), 0, self.W)
        df['y_start'] = np.clip(np.digitize(df['miny'].values, self.lat_bins) - 1, 0, self.H - 1)
        df['y_end'] = np.clip(np.digitize(df['maxy'].values, self.lat_bins), 0, self.H)

        valid_mask = (df['x_end'] > df['x_start']) & (df['y_end'] > df['y_start'])
        df = df[valid_mask].reset_index(drop=True)

        return df

    def _get_time_index(self, begin_year: int, end_year: int, subset: str) -> pd.DatetimeIndex:
        """Generate time index based on subset"""
        if subset == 'train':
            return pd.date_range(f"{begin_year}-01-01", f"{end_year}-12-31", freq='D')
        elif subset == 'val':
            return pd.date_range(f"{begin_year}-01-01", f"{end_year}-06-30", freq='D')
        else:
            return pd.date_range(f"{begin_year}-07-01", f"{end_year}-12-31", freq='D')


# ============================================================================
# VISUALIZATION
# ============================================================================
class Visualizer:
    """Handle map plotting and visualization"""

    def __init__(self, config: Config):
        self.config = config
        self.vn_adm1 = None
        self.dbscl = None

    def load_shapefile(self):
        """Load Vietnam administrative boundaries"""
        if self.vn_adm1 is None:
            self.vn_adm1 = gpd.read_file(self.config.vn_adm1_path)
            self.dbscl = self.vn_adm1[self.vn_adm1["NAME_1"].isin(self.config.mekong_provinces)]

    def plot_map(self, x: np.ndarray, title: str = "Rainfall Prediction Map", cmap: str = 'turbo'):
        """Plot spatial data on map"""
        self.load_shapefile()

        extent = [self.config.min_lon, self.config.max_lon, self.config.min_lat, self.config.max_lat]

        fig, ax = plt.subplots(figsize=(10, 10))

        self.vn_adm1.boundary.plot(ax=ax, color='lightgrey', linewidth=0.4)
        self.dbscl.boundary.plot(ax=ax, color='black', linewidth=1)

        im = ax.imshow(x, extent=extent, origin='lower', cmap=cmap, alpha=0.8)

        plt.colorbar(im, ax=ax, label="Rainfall prediction (mm/h)")
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.set_xlim(self.config.min_lon, self.config.max_lon)
        ax.set_ylim(self.config.min_lat, self.config.max_lat)
        plt.tight_layout()
        plt.show()


# ============================================================================
# PIPELINE
# ============================================================================
class MekongDataPipeline:
    """Main pipeline orchestrator"""

    def __init__(self, config: Config = None):
        self.config = config or Config()
        self.loader = DataLoader(self.config)
        self.splitter = DataSplitter(self.config)
        self.processor = MekongDataProcessor(self.config)
        self.tensor_builder = TensorBuilder(self.config)
        self.visualizer = Visualizer(self.config)

    def run(self) -> Dict:
        """Execute full pipeline"""
        # Load data
        data = self.loader.load_all_data()

        # Split data
        splits = self.splitter.split_all(data)

        # Process data
        print("\nProcessing data...")
        processed = self._process_all_splits(splits)

        # Build tensors
        print("\nBuilding tensors...")
        tensors = self._build_all_tensors(processed)

        return {
            'data': data,
            'splits': splits,
            'processed': processed,
            'tensors': tensors
        }

    def _process_all_splits(self, splits: Dict) -> Dict:
        """Process all data splits"""
        result = {}

        mekong_train, mekong_val, mekong_test = splits['mekong']
        mekong_train, mekong_val, mekong_test = self.processor.robust_scaling(
            mekong_train, mekong_val, mekong_test
        )
        mekong_train, mekong_val, mekong_test = self.processor.apply_pca_groups(
            mekong_train, mekong_val, mekong_test
        )
        result['mekong'] = (mekong_train, mekong_val, mekong_test)

        target_train, target_val, target_test = splits['target']
        result['target'] = self.processor.robust_scaling(
            target_train, target_val, target_test, target=True
        )

        s1_train, s1_val, s1_test = splits['sentinel_1']
        result['sentinel_1'] = self.processor.robust_scaling(s1_train, s1_val, s1_test)

        s2_train, s2_val, s2_test = splits['sentinel_2']
        result['sentinel_2'] = self.processor.robust_scaling(s2_train, s2_val, s2_test)

        return result

    def _build_all_tensors(self, processed: Dict) -> Dict:
        """Build all tensors"""
        result = {}

        mekong_train, _, _ = processed['mekong']
        target_train, _, _ = processed['target']
        s1_train, _, _ = processed['sentinel_1']
        s2_train, _, _ = processed['sentinel_2']

        features = {
            'mekong': [c for c in mekong_train.columns if c not in self.config.exclude_cols],
            'target': [c for c in target_train.columns if c not in self.config.exclude_cols],
            'sentinel_1': [c for c in s1_train.columns if c not in self.config.exclude_cols],
            'sentinel_2': [c for c in s2_train.columns if c not in self.config.exclude_cols + ['AREA']]
        }

        # Build Mekong tensors
        print("Building Mekong tensors...")
        for split_name, split_data in [('train', 0), ('val', 1), ('test', 2)]:
            result[f'mekong_{split_name}'] = self.tensor_builder.build_mekong_tensor(
                processed['mekong'][split_data], features['mekong']
            )
            result[f'target_{split_name}'] = self.tensor_builder.build_mekong_tensor(
                processed['target'][split_data], features['target']
            )

        # Build Sentinel tensors
        train_start = pd.Timestamp(self.config.train_start)
        val_start = pd.Timestamp(self.config.val_start)
        test_start = pd.Timestamp(self.config.test_start)
        train_end = pd.Timestamp(self.config.train_end)
        val_end = pd.Timestamp(self.config.val_end)
        test_end = pd.Timestamp(self.config.test_end)

        # Sentinel-1
        result['sentinel_1_train'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_1'][0], features['sentinel_1'],
            train_start.year, train_end.year, 'train'
        )
        result['sentinel_1_val'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_1'][1], features['sentinel_1'],
            val_start.year, val_end.year, 'val'
        )
        result['sentinel_1_test'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_1'][2], features['sentinel_1'],
            test_start.year, test_end.year, 'test'
        )

        # Sentinel-2
        result['sentinel_2_train'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_2'][0], features['sentinel_2'],
            train_start.year, train_end.year, 'train'
        )
        result['sentinel_2_val'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_2'][1], features['sentinel_2'],
            val_start.year, val_end.year, 'val'
        )
        result['sentinel_2_test'] = self.tensor_builder.build_sentinel_tensor(
            processed['sentinel_2'][2], features['sentinel_2'],
            test_start.year, test_end.year, 'test'
        )

        return result

def minmax_scale_tensor(train, val, test, target=False):
    """
    MinMax scale dữ liệu 4D [samples, H, W, C] per-channel.

    Args:
        train, val, test: np.ndarray [samples, H, W, C]
        target (bool): True nếu đây là target, sẽ lưu scaler ra file "minmax_scaler_target.pkl"

    Returns:
        train_scaled, val_scaled, test_scaled
    """
    n_channels = train.shape[-1]
    train_scaled = np.zeros_like(train, dtype=np.float32)
    val_scaled   = np.zeros_like(val, dtype=np.float32)
    test_scaled  = np.zeros_like(test, dtype=np.float32)

    for c in range(n_channels):
        scaler = MinMaxScaler()

        # Flatten thành 2D [num_pixels, 1]
        train_flat = train[..., c].reshape(-1, 1)
        val_flat   = val[..., c].reshape(-1, 1)
        test_flat  = test[..., c].reshape(-1, 1)

        # Fit scaler trên train
        scaler.fit(train_flat)

        # Transform tất cả data
        train_scaled[..., c] = scaler.transform(train_flat).reshape(train.shape[0], train.shape[1], train.shape[2])
        val_scaled[..., c]   = scaler.transform(val_flat).reshape(val.shape[0], val.shape[1], val.shape[2])
        test_scaled[..., c]  = scaler.transform(test_flat).reshape(test.shape[0], test.shape[1], test.shape[2])

        # Lưu scaler nếu là target
        if target:
            joblib.dump(scaler, "minmax_scaler_target.pkl")

    return train_scaled, val_scaled, test_scaled

def save_timeseries_tensor(train_data, val_data, test_data,
                           train_target, val_target, test_target,
                           base_path="dataset_ts"):
    """
    Lưu dataset time-series thành tensor 3D riêng lẻ, per-sample.

    Args:
        train_data, val_data, test_data: [samples, H, W, C]
        train_target, val_target, test_target: [samples, H, W, 1]
        base_path: thư mục gốc
    """
    sets = {
        "train": (train_data, train_target),
        "val":   (val_data, val_target),
        "test":  (test_data, test_target)
    }

    for set_name, (data, target) in sets.items():
        feature_dir = os.path.join(base_path, set_name, "features")
        target_dir  = os.path.join(base_path, set_name, "target")
        os.makedirs(feature_dir, exist_ok=True)
        os.makedirs(target_dir, exist_ok=True)

        n_samples = data.shape[0]

        for i in range(n_samples):
            feat_path = os.path.join(feature_dir, f"{i:04d}.npy")
            targ_path = os.path.join(target_dir,  f"{i:04d}.npy")

            np.save(feat_path, data[i])   # [H, W, C]
            np.save(targ_path, target[i]) # [H, W, 1]

    print(f"Lưu xong dataset time-series vào {base_path}!")

## Run

In [ ]:
# Initialize configuration
config = Config(
    chunk_size=50,  # Optimized chunk size
    use_multiprocessing=True,
    n_workers=None  # Auto-detect CPU cores
)

# Run pipeline
pipeline = MekongDataPipeline(config)
results = pipeline.run()
joblib.dump(pipeline.processor.robust_scaler_target, "robust_scaler_target.pkl")

# Access results
tensors = results['tensors']
print("\nTensor shapes:")
for name, tensor in tensors.items():
    print(f"{name:20s}: {tensor.shape}")

import torch

train_data = torch.cat((
    torch.from_numpy(tensors['mekong_train']).float(),
    torch.from_numpy(tensors['sentinel_1_train']).float(),
    torch.from_numpy(tensors['sentinel_2_train']).float()
), dim=-1)

val_data = torch.cat((
    torch.from_numpy(tensors['mekong_val']).float(),
    torch.from_numpy(tensors['sentinel_1_val']).float(),
    torch.from_numpy(tensors['sentinel_2_val']).float()
), dim=-1)

test_data = torch.cat((
    torch.from_numpy(tensors['mekong_test']).float(),
    torch.from_numpy(tensors['sentinel_1_test']).float(),
    torch.from_numpy(tensors['sentinel_2_test']).float()
), dim=-1)


# =============================
# 🎯 Gán target tương ứng
# =============================

train_target = torch.from_numpy(tensors['target_train'])
val_target = torch.from_numpy(tensors['target_val'])
test_target = torch.from_numpy(tensors['target_test'])


train_data_scale, val_data_scale, test_data_scale = minmax_scale_tensor(train_data, val_data, test_data)

train_target_scale, val_target_scale, test_target_scale = minmax_scale_tensor(train_target, val_target, test_target, target=True)

save_timeseries_tensor(train_data_scale, val_data_scale, test_data_scale,
                       train_target_scale, val_target_scale, test_target_scale,
                       base_path="dataset_ts")


Loading data files in parallel...
Splitting data into train/val/test...
mekong       - Train: (23738, 27), Val: (2366, 27), Test: (2392, 27)
sentinel_1   - Train: (1890, 10), Val: (142, 10), Test: (146, 10)
sentinel_2   - Train: (2649, 10), Val: (361, 10), Test: (129, 10)
target       - Train: (23738, 7), Val: (2366, 7), Test: (2392, 7)

Processing data...
Applying PCA to feature groups...

Building tensors...
Building Mekong tensors...
Building Sentinel tensor (train) with multiprocessing...


Processing train: 100%|██████████| 37/37 [00:16<00:00,  2.20it/s]

Building Sentinel tensor (val) with multiprocessing...



Processing val: 100%|██████████| 4/4 [00:02<00:00,  1.95it/s]

Building Sentinel tensor (test) with multiprocessing...


...

Building Sentinel tensor (val) with multiprocessing...



Processing val: 100%|██████████| 4/4 [00:01<00:00,  2.88it/s]

Building Sentinel tensor (test) with multiprocessing...



Processing test: 100%|██████████| 4/4 [00:01<00:00,  2.04it/s]


Tensor shapes:
mekong_train        : (1826, 35, 35, 15)
target_train        : (1826, 35, 35, 1)
mekong_val          : (182, 35, 35, 15)
target_val          : (182, 35, 35, 1)
mekong_test         : (184, 35, 35, 15)
target_test         : (184, 35, 35, 1)
sentinel_1_train    : (1826, 35, 35, 5)
sentinel_1_val      : (182, 35, 35, 5)
sentinel_1_test     : (184, 35, 35, 5)
sentinel_2_train    : (1826, 35, 35, 4)
sentinel_2_val      : (182, 35, 35, 4)
sentinel_2_test     : (184, 35, 35, 4)
